# Lab 12B: A tiny Vision Transformer

This is the second notebook of Lab 12 (see 12A for an overview).

We build a small Vision Transformer (ViT) following **Lecture 12, Code 5**, train it on the cat, dog and
horse images of CIFAR-10 (32 × 32 pixels, the same three classes as in Lab 2), and compare it with a small CNN.

| Part | You will… |
|---|---|
| 1. Images as sequences of patches | see how an image becomes a sequence of tokens |
| 2. The ViT | fill in the three lines that make a Transformer a *Vision* Transformer: CLS token, position embeddings, classification from the CLS token |
| 3. ViT versus CNN | train both on 1,500 and on 15,000 images: who needs more data? |
| 4. What does the CLS token look at? | plot the attention of the CLS token over the patches |
| 5. Exam-style questions | check your understanding |

**Before you start: Runtime → Change runtime type → GPU.** Training takes about 3 minutes on a GPU (about an hour on a CPU).

## Setup

In [ ]:
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

import os
import zipfile
from pathlib import Path

import gdown
from PIL import Image

# The cat, dog and horse images of CIFAR-10 (32 × 32 pixels), downloaded from Google Drive:
# train/{cat,dog,horse}: 5,000 images each; test/{cat,dog,horse}: 1,000 images each
CLASSES = ['cat', 'dog', 'horse']

if not Path('train').exists():
    if not Path('cifar_cat_dog_horse.zip').exists():
        file_id = '1wvqLLFioimWDLfl7oqzCZi-q-tKDfXVg'
        gdown.download(f'https://drive.google.com/uc?id={file_id}', 'cifar_cat_dog_horse.zip', quiet=False)
    with zipfile.ZipFile('cifar_cat_dog_horse.zip') as z:
        z.extractall('.')


def load_split(split):
    xs, ys = [], []
    for label, name in enumerate(CLASSES):
        files = sorted(Path(split, name).glob('*.jpg'))
        xs += [np.array(Image.open(f).convert('RGB')) for f in files]
        ys += [label] * len(files)
    x = torch.tensor(np.stack(xs)).permute(0, 3, 1, 2).float() / 255   # (N, 3, 32, 32) in [0, 1]
    y = torch.tensor(ys)
    perm = torch.randperm(len(y), generator=torch.Generator().manual_seed(0))   # mix the classes
    return x[perm], y[perm]


X_train, Y_train = load_split('train')
X_test, Y_test = load_split('test')
MEAN = X_train.mean((0, 2, 3), keepdim=True)
STD = X_train.std((0, 2, 3), keepdim=True)
print(f'{len(X_train)} training images, {len(X_test)} test images')


def augment(x):
    # Random horizontal flips and random shifts of up to 4 pixels (the same for both models)
    flip = torch.rand(x.shape[0], 1, 1, 1, device=x.device) < 0.5
    x = torch.where(flip, x.flip(3), x)
    x = F.pad(x, (4, 4, 4, 4), mode='reflect')
    i, j = torch.randint(0, 9, (2,))
    return x[:, :, i:i + 32, j:j + 32]


def train(model, n_train, steps=1500, lr=1e-3, seed=0):
    # Trains on the first n_train images (in random order) for a fixed number of steps; returns the test accuracy
    set_seed(seed)
    model.to(device)
    idx = torch.randperm(len(X_train))[:n_train]
    X, Y = X_train[idx].to(device), Y_train[idx].to(device)
    mean, std = MEAN.to(device), STD.to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    schedule = torch.optim.lr_scheduler.OneCycleLR(optimiser, lr, total_steps=steps)
    t0 = time.time()
    model.train()
    for step in range(steps):
        b = torch.randint(0, n_train, (128,), device=device)
        loss = F.cross_entropy(model((augment(X[b]) - mean) / std), Y[b])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        schedule.step()
    acc = evaluate(model)
    print(f'  {type(model).__name__} on {n_train} images: test accuracy {acc:.0%}  ({time.time() - t0:.0f} s)')
    return acc


@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    for k in range(0, len(X_test), 500):
        x = ((X_test[k:k + 500] - MEAN) / STD).to(device)
        correct += (model(x).argmax(1).cpu() == Y_test[k:k + 500]).sum().item()
    return correct / len(X_test)


count = lambda m: sum(p.numel() for p in m.parameters())

---
# Part 1 — Images as sequences of patches

A Transformer works on a sequence of tokens. ViT turns an image into one by cutting it into non-overlapping
$P \times P$ patches (Lecture 12, Section 5.1):

$$N = \frac{H \times W}{P^2} \text{ patches} = N \text{ tokens}$$

Each patch is flattened and projected to a $d$-dimensional token by the same linear layer. In code, a
`Conv2d` with `kernel_size = stride = P` does both at once: each output position sees exactly one patch.

Below, an image cut into patches of $4 \times 4$ pixels.

In [ ]:
P = 4
img = X_train[1]
n = 32 // P
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5), gridspec_kw={'width_ratios': [1, 1.2]})
axs[0].imshow(img.permute(1, 2, 0)); axs[0].set_title(f'image: {CLASSES[Y_train[1]]}, 32 × 32'); axs[0].axis('off')
grid = torch.ones(3, n * (P + 1), n * (P + 1))
for r in range(n):
    for c in range(n):
        grid[:, r * (P + 1):r * (P + 1) + P, c * (P + 1):c * (P + 1) + P] = img[:, r * P:(r + 1) * P, c * P:(c + 1) * P]
axs[1].imshow(grid.permute(1, 2, 0)); axs[1].set_title(f'{n * n} patches of {P} × {P}: the tokens'); axs[1].axis('off')
plt.show()

### Questions 1

1. How many tokens does a $32 \times 32$ image give with $P = 4$? With $P = 8$? How large is the attention matrix in each case?
2. Each patch is flattened to a vector before the linear projection. How long is that vector for $P = 4$ and an RGB image?

---
# Part 2 — The ViT

The Transformer encoder block is given: it is **Lecture 12, Code 3** (pre-norm self-attention and
feed-forward network, each with a residual connection), with one addition: it keeps its attention weights in
`self.attention_weights`, so we can plot them in Part 4.

In [ ]:
class TransformerEncoderBlock(nn.Module):
    # Lecture 12, Code 3 (pre-norm), keeping the attention weights for Part 4
    def __init__(self, d_model, n_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(ffn_dim, d_model), nn.Dropout(dropout))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, weights = self.attn(x_norm, x_norm, x_norm, average_attn_weights=False)
        self.attention_weights = weights.detach()      # (B, heads, tokens, tokens)
        x = x + self.dropout(attn_out)                 # residual connection
        x = x + self.dropout(self.ffn(self.norm2(x)))  # residual connection
        return x

### 🛠️ TASK 1: The ViT forward pass

The ViT adds three things to a plain Transformer encoder (Lecture 12, Sections 5.1–5.2):

1. a learnable **CLS token**, put *in front of* the patch tokens; after the encoder, it holds a summary of the
   whole image;
2. learnable **position embeddings**, one per token (the CLS token included), *added* to the tokens;
3. a classification head that looks **only at the CLS token's output**.

**🔗 Theory ↔ code**

| Lecture | Code | Shape |
|---|---|---|
| the patch tokens | `x` after the patch embedding | (B, N, d) |
| prepend the CLS token to the sequence | `torch.cat([cls, x], dim=1)` | (B, N+1, d) |
| add the position embeddings | `x + self.pos_embed` | (B, N+1, d) |
| the output of the CLS token (token 0) | `x[:, 0]` | (B, d) |
| classify it | `self.head(...)` | (B, n_classes) |

In [ ]:
class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, n_classes=3, d_model=128, n_heads=4, n_layers=4, ffn_dim=256):
        super().__init__()
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)  # patches -> tokens
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, d_model))              # +1 for the CLS token
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerEncoderBlock(d_model, n_heads, ffn_dim) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)   # (B, d, H/P, W/P) -> (B, N, d): one token per patch
        cls = self.cls_token.expand(x.shape[0], -1, -1)      # (B, 1, d): the same CLS token for every image
        x = ???                                              # 1. prepend the CLS token     -> (B, N+1, d)
        x = ???                                              # 2. add position embeddings   -> (B, N+1, d)
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return ???                                           # 3. classify the CLS token    -> (B, n_classes)

In [ ]:
# Test for Task 1
_m = ViT().eval()
_x = torch.randn(2, 3, 32, 32)
with torch.no_grad():
    _out = _m(_x)
    assert _out.shape == (2, 3), f'the output should have shape (2, 3); got {tuple(_out.shape)}'
    assert _m.blocks[0].attention_weights.shape == (2, 4, 65, 65), 'the sequence should have 64 patch tokens + 1 CLS token = 65 tokens'
    # the position embeddings must be used
    _noise = torch.randn_like(_m.pos_embed)
    _m.pos_embed.add_(_noise)
    assert not torch.allclose(_m(_x), _out), 'the output does not change when the position embeddings change: are they added?'
    _m.pos_embed.sub_(_noise)
    # the head must see the CLS token (token 0) only
    _t = _m.patch_embed(_x).flatten(2).transpose(1, 2)
    _t = torch.cat([_m.cls_token.expand(2, -1, -1), _t], 1) + _m.pos_embed
    for _b in _m.blocks:
        _t = _b(_t)
    assert torch.allclose(_out, _m.head(_m.norm(_t)[:, 0]), atol=1e-5), 'the head should classify the output of the CLS token, x[:, 0]'
print('✅ Task 1 passed')
print(f'ViT parameters: {count(ViT()):,}')

---
# Part 3 — ViT versus CNN: who needs more data?

The lecture says that a CNN's **inductive biases** (locality, translation equivariance) are "free
regularisation" when data is limited, while a ViT has to learn such structure from the data (Sections 1 and 6).
Let's test that on a small scale. The CNN below has three blocks of two convolutions each, with a parameter
count in the same range as our ViT.

We train both models on **1,500** and on **15,000** training images (10% and 100% of the cat/dog/horse
images), each for the same number of steps, with the same augmentation, optimiser and schedule.

✍️ **Prediction:** sketch the test accuracy of both models for the two training-set sizes.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                         nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(), nn.MaxPool2d(2))


class CNN(nn.Module):
    def __init__(self, n_classes=3, c=44):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, c), conv_block(c, 2 * c), conv_block(2 * c, 4 * c))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.1), nn.Linear(4 * c, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


print(f'CNN parameters: {count(CNN()):,}')
print(f'ViT parameters: {count(ViT()):,}')

In [ ]:
SIZES = [1500, 15000]
results = {'CNN': [], 'ViT': []}
for n in SIZES:
    cnn, vit = CNN(), ViT()
    results['CNN'].append(train(cnn, n, lr=3e-3))
    results['ViT'].append(train(vit, n, lr=1e-3))
vit_model = vit                       # the ViT trained on all 15,000 images: we use it in Part 4

In [ ]:
x = np.arange(len(SIZES))
plt.figure(figsize=(6, 4))
for k, (name, acc) in enumerate(results.items()):
    bars = plt.bar(x + (k - 0.5) * 0.35, acc, 0.35, label=name)
    plt.bar_label(bars, [f'{a:.0%}' for a in acc])
plt.axhline(1 / 3, color='grey', ls='--', label='chance')
plt.xticks(x, [f'{n:,} training images' for n in SIZES]); plt.ylabel('test accuracy'); plt.ylim(0, 1)
plt.title('ViT versus CNN, trained from scratch'); plt.legend(loc='lower right'); plt.show()

### Questions 3

1. Which model is better with 1,500 training images? With 15,000? Does the gap between them grow or shrink with more data?
2. Explain the result with the inductive biases of the CNN. What does the ViT have to learn from the data that the CNN gets "for free"?
3. The lecture says that ViTs beat CNNs when they are pre-trained on very large datasets (ImageNet-21k, JFT-300M).
   What would you expect if we trained both models on 15 million instead of 15,000 images, and why?
4. In practice, how would you use a ViT for a small dataset such as your project's? (Hint: Lab 5.)

---
# Part 4 — What does the CLS token look at?

The CLS token collects information from the patches through attention. In the last block, its attention
weights over the 64 patches show which parts of the image it reads from. We take the ViT trained on all 15,000
images in Part 3 and plot, for a few test images, the CLS token's attention per head (as an $8 \times 8$ map: one value per patch).

In [ ]:
@torch.no_grad()
def cls_attention(model, x):
    model.eval()
    model(((x - MEAN) / STD).to(device))
    w = model.blocks[-1].attention_weights        # (B, heads, 65, 65)
    return w[:, :, 0, 1:].reshape(x.shape[0], -1, 8, 8).cpu()   # from the CLS token (row 0) to the 64 patches


idx = [0, 3, 5, 7, 11, 12]
maps = cls_attention(vit_model, X_test[idx])
n_heads = maps.shape[1]
fig, axs = plt.subplots(len(idx), n_heads + 1, figsize=(1.8 * (n_heads + 1), 1.9 * len(idx)))
for r, i in enumerate(idx):
    axs[r, 0].imshow(X_test[i].permute(1, 2, 0)); axs[r, 0].set_title(CLASSES[Y_test[i]], fontsize=9)
    for h in range(n_heads):
        axs[r, h + 1].imshow(X_test[i].permute(1, 2, 0))
        axs[r, h + 1].imshow(F.interpolate(maps[r:r + 1, h:h + 1], size=32, mode='nearest')[0, 0], cmap='jet', alpha=0.5)
        if r == 0:
            axs[r, h + 1].set_title(f'head {h}', fontsize=9)
for ax in axs.flat:
    ax.axis('off')
plt.suptitle('Last block: attention of the CLS token over the patches'); plt.tight_layout(); plt.show()

### Questions 4

1. Do the heads look at the animal, at the background, or everywhere? Do different heads look at different things?
2. These maps come from the **last** block. Could a patch in the **first** block already attend to a patch in the
   opposite corner of the image? Could a $3 \times 3$ convolution in the first layer of a CNN? (Lecture 12, Section 5.3.)
3. Compare these maps with the Grad-CAM maps from Lab 10. What do attention maps show, and what do they *not* show?

---
# Part 5 — Exam-style questions

**Q1.** A $96 \times 96$ RGB image is split into patches of $8 \times 8$ pixels for a ViT. How many tokens enter the Transformer, including the CLS token?

* (a) 144
* (b) 145
* (c) 1,152
* (d) 9,216

**Q2.** In ViT, which output is passed to the classification head?

* (a) The average of all patch tokens.
* (b) The output of the CLS token.
* (c) The output of the last patch token.
* (d) All token outputs, concatenated.

**Q3.** A ViT and a ResNet of similar size are trained from scratch on a small dataset. Which outcome is most likely, and why?

* (a) The ViT wins, because attention has a global receptive field from the first layer.
* (b) Both perform the same, because they have the same number of parameters.
* (c) The ResNet wins, because its locality and weight sharing are built-in assumptions that the ViT must learn from data.
* (d) The ViT wins, because it has no inductive biases that could be wrong.

**Q4.** Why are position embeddings added to the patch tokens of a ViT?

* (a) Because self-attention by itself does not depend on the order of the tokens, so without them the model could not tell where a patch came from.
* (b) To normalise the tokens.
* (c) To reduce the number of tokens.
* (d) To replace the CLS token.